# 01 · Prices, Returns & Volatility
**Goal:** turn a price series into the two numbers quants actually model — *returns* and *volatility* — and see why we annualize with √time.

No finance background needed. Run each cell top to bottom (Shift+Enter). Change the numbers and re-run — that's the point.

> Maps to: every project. See `QUANT_KNOWLEDGE_BASE.md` §1.1–1.3.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

## 1. A price series
A **price** is what one unit of an asset costs at a moment in time. A list of prices over time is a *time series*. Let's make a fake 1-year daily series so we don't need the internet.

In [ ]:
dates = pd.bdate_range('2024-01-01', periods=252)  # ~252 trading days in a year
# a gently drifting, wiggly price
steps = rng.normal(0.0005, 0.01, len(dates))
price = pd.Series(100 * np.cumprod(1 + steps), index=dates, name='price')
price.plot(title='A fake daily price series', figsize=(9,3)); plt.ylabel('price'); plt.show()
price.head()

## 2. Returns: simple vs log
We don't model prices directly (they trend and wander). We model **returns** — the percentage change from one day to the next.

- **Simple return:** `P_t / P_{t-1} - 1`  → these *add across assets* (use for portfolios).
- **Log return:** `ln(P_t / P_{t-1})`     → these *add across time* (use for modeling).

For small daily moves they're nearly identical.

In [ ]:
simple = price.pct_change().dropna()
logret = np.log(price / price.shift(1)).dropna()
compare = pd.DataFrame({'simple': simple, 'log': logret}).head()
print(compare)
print('\nmax abs difference between them:', float((simple - logret).abs().max()))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11,3))
simple.plot(ax=ax[0], title='daily simple returns'); ax[0].axhline(0, color='k', lw=.5)
ax[1].scatter(simple, logret, s=6); ax[1].set(title='simple vs log (nearly a line)',
             xlabel='simple', ylabel='log')
plt.tight_layout(); plt.show()

## 3. Volatility = standard deviation of returns
**Volatility** is how much returns bounce around — the *standard deviation* of returns. It is the single most important number in risk. Bigger = more uncertain.

In [ ]:
daily_vol = simple.std()
print(f'daily volatility: {daily_vol:.4f}  ({daily_vol*100:.2f}% per day)')

## 4. Annualizing with √time
Volatility is usually quoted *per year*. Variance grows linearly with time, so standard deviation grows with the **square root of time**. With ~252 trading days per year:

$$\sigma_{annual} = \sigma_{daily}\times\sqrt{252}$$

In [ ]:
annual_vol = daily_vol * np.sqrt(252)
print(f'annualized volatility: {annual_vol:.4f}  ({annual_vol*100:.1f}% per year)')

# Why sqrt? Watch volatility grow with the sqrt of the horizon:
for h in [1, 5, 21, 252]:
    print(f'{h:3d}-day vol ~= {daily_vol*np.sqrt(h):.4f}')

## 5. More examples: comparing two assets
Volatility is what makes one asset 'riskier' than another. Let's build a calm asset and a wild one and compare their numbers and paths.

In [ ]:
calm = pd.Series(100*np.cumprod(1+rng.normal(0.0003, 0.006, 252)), index=dates)
wild = pd.Series(100*np.cumprod(1+rng.normal(0.0003, 0.025, 252)), index=dates)
for name, p in [('calm', calm), ('wild', wild)]:
    r = p.pct_change().dropna()
    print(f'{name:5s}  daily vol {r.std()*100:5.2f}%   annual vol {r.std()*np.sqrt(252)*100:5.1f}%')
pd.DataFrame({'calm': calm, 'wild': wild}).plot(figsize=(9,3), title='Same drift, different volatility'); plt.show()

### 🧪 Try it yourself
1. Change the `0.01` in `steps` to `0.03` — daily and annual vol roughly triple.
2. Compute the **total** return two ways: `price.iloc[-1]/price.iloc[0]-1` (simple) and `logret.sum()` then `np.exp(...)-1` (log). They match — that's log returns adding across time.
3. Re-run with a different seed and see how much the *estimated* vol wobbles — volatility itself is uncertain.

**You should see:** the simple and log returns are almost the same line; daily vol is ~1%; annual vol is ~15–20%. Multi-horizon vol scales with √h, not h.

### In the projects
- `pct_change()` (simple returns) → project **07** `src/data.py:portfolio_returns`.
- annual→per-interval vol via √time → project **29** `src/data.py:ExecutionProblem.sigma_step`.
- log returns for bar statistics → project **30** `src/bars.py:bar_return_stats`.